# Homework 3 - Text generation with LSTM and Transformer networks


## Installs the unidecode library and downloads the Shakespeare dataset.

In [1]:
!pip install unidecode
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-26 07:12:13--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  6.33MB/s    in 0.2s    

2026-04-26 07:12:14 (6.33 MB/s) - ‘input.txt’ saved [1115394/1115394]



## LSTM implementation

For this task you will implement the LSTM neural network architecture and train it on the task of character-level text generation. Implement a single layer LSTM and optionally extend your implementation to multiple layers to generate better results.

Links:

- https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html -- Lists the equations for each component of the LSTM cell.
- http://colah.github.io/posts/2015-08-Understanding-LSTMs/ -- Intuitive explanation of LSTM
- http://karpathy.github.io/2015/05/21/rnn-effectiveness/ -- Explanation and uses of RNNs.


Implement the initialization and the forward pass of a LSTMCell and use it as part of the LSTMSimple network class.

The input of the LSTM network will be a sequence of characters, whereas the input of the LSTMCell will be a single input character (x), the output of the previous iteration (C) and the hidden state of the previous iteration (h). Iteratively process the entire input character sequence and calculate the loss based on the prediction at each time step.

### Do NOT use the torch.nn.LSTM class.


In [2]:
# shared functions for training and sampling
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset
import unidecode
import string
import random
from torch.autograd import Variable
import math


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def random_seed_text(dataset, seed_len=40):
    start = np.random.randint(0, dataset.file_len - seed_len - 1)
    return dataset.file[start:start + seed_len]

def ids_to_text(ids, dataset):
    if torch.is_tensor(ids):
        ids = ids.detach().cpu().long().flatten().tolist()
    return "".join(dataset.all_characters[int(i)] for i in ids)

def sample_next_index(logits, method="topk", top_k=5):
    if logits.dim() == 1:
        logits = logits.unsqueeze(0)

    if method == "greedy":
        return torch.argmax(logits, dim=1)

    if method == "topk":
        k = min(top_k, logits.size(-1))
        values, indices = torch.topk(logits, k, dim=1)
        probs = F.softmax(values, dim=1)
        chosen = torch.multinomial(probs, num_samples=1)
        return indices.gather(1, chosen).squeeze(1)


def run_experiments(train_fn, experiment_config):
    results = {}
    for chunk_len, params in experiment_config.items():
        results[chunk_len] = train_fn(chunk_len=chunk_len, **params)
    return results

def compare_trained_runs(results, sample_fn, title, num_examples=5, seed_len=40, gen_len=100):
    sequence_lengths = list(results.keys())
    reference_dataset = results[sequence_lengths[0]]["dataset"]
    seed_texts = [random_seed_text(reference_dataset, seed_len) for _ in range(num_examples)]

    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)

    for i, seed_text in enumerate(seed_texts, start=1):
        print("\n" + "#" * 120)
        print(f"Example {i}")
        print("#" * 120)
        print("\nInput text:")
        print(seed_text)

        for seq_len in sequence_lengths:
            run = results[seq_len]
            print("\n" + "-" * 100)
            print(f"Sequence length = {seq_len}")
            print("-" * 100)

            print("\nTop-k sampling:")
            print(sample_fn(run["model"], run["dataset"], seed_text, gen_len, method="topk", chunk_len=seq_len))

            print("\nGreedy sampling:")
            print(sample_fn(run["model"], run["dataset"], seed_text, gen_len, method="greedy", chunk_len=seq_len))


In [ ]:
class LSTMCell(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):

        super(LSTMCell, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # linear layer to compute all 4 gates at once
        # forget gate, input gate, candidate gate, output gate
        self.x2h = nn.Linear(input_dim, 4 * hidden_dim)
        self.h2h = nn.Linear(hidden_dim, 4 * hidden_dim) 

    def forward(self, x, C, h):
        # x - batch of encoded characters
        # C - cell state of the previous iteration
        # h - hidden state of the previous iteration

        # combined affine transform 
        gates = self.x2h(x) + self.h2h(h)

        # split into 4 parts 
        f_t, i_t, g_t, o_t = torch.chunk(gates, 4, dim=1)

        # apply activations
        f_t = torch.sigmoid(f_t) # forget gate
        i_t = torch.sigmoid(i_t) # input gate
        g_t = torch.tanh(g_t)  # candidate cell state
        o_t = torch.sigmoid(o_t) # output gate

        # update cell state and hidden state
        C_out = f_t * C + i_t * g_t
        h_out = o_t * torch.tanh(C_out)

        return C_out, h_out

class LSTMSimple(nn.Module):
    def __init__(self, seq_length, input_dim, hidden_dim, output_dim, batch_size):
        super(LSTMSimple, self).__init__()

        self.seq_length = seq_length
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.batch_size = batch_size

        # initialize one LSTM cell
        self.lstm_cell = LSTMCell(input_dim, hidden_dim, output_dim)

        # project hidden state to output vocabulary size
        self.proj = nn.Linear(hidden_dim, output_dim)


    def forward(self, x):

        batch_size = x.size(0)
        seq_len = x.size(1)
        device = x.device

        # initialize hidden state and cell state to zeros
        h = torch.zeros(batch_size, self.hidden_dim, device=device)
        C = torch.zeros(batch_size, self.hidden_dim, device=device)

        outputs = []

        # process sequence one timestep at a time
        for t in range(seq_len):
            x_t = x[:, t, :]           
            C, h = self.lstm_cell(x_t, C, h)
            # predict next character from hidden state
            out_t = self.proj(h)      
            outputs.append(out_t.unsqueeze(1))

        # concatenate outputs across time
        outputs = torch.cat(outputs, dim=1)  

        return outputs, (C, h)


### LSTM Sampling Code

To generate text the network must predict the next character in a sequence, however networks do not produce a single character but rather estimate the likelihood for each possible character. Sampling characters from the network output can be done in different ways with common ones being the Greedy sampling process and Top-K sampling.

In the simple greedy sampling method the network takes a text prompt as input and generates an additional N tokens by always taking the token with the highest prediction score as the next token.

In the Top-K sampling, randomness is added to the sampling process as the network samples from K most likely predicitons at each step. This alleviates the problem of generative models repeating text but may generate incorrect text by sampling inappropriate tokens.


In [ ]:
def greedy_sampling_lstm(lstm, x, num_chars):
    # x -- b x onehot_char
    outputs = torch.zeros((1,num_chars,x.shape[2]))
    t_outputs, (cell_state, hidden) = lstm(x.float())
    for c in range(num_chars):
        output_tmp = torch.softmax(lstm.proj(hidden),dim=1)
        top_ind = torch.argmax(output_tmp,dim=1)[0]
        tmp = torch.zeros_like(x[:,0,:]).cuda()
        tmp[:,top_ind] = 1
        outputs[:,c] = tmp

        cell_state, hidden = lstm.lstm_cell(tmp,cell_state,hidden)
    return outputs

def topk_sampling_lstm(lstm, x, num_chars):
    # x -- b x onehot_char
    outputs = torch.zeros((1,num_chars,x.shape[2]))
    t_outputs, (cell_state, hidden) = lstm(x.float())
    for c in range(num_chars):
        output_vals, output_ind = torch.topk(lstm.proj(hidden), 5, dim=1)
        output_tmp = torch.softmax(output_vals,dim=1)
        top_ind = torch.multinomial(output_tmp[0], 1)[0]
        tmp = torch.zeros_like(x[:,0,:]).cuda()
        tmp[:,output_ind[0,top_ind]] = 1
        outputs[:,c] = tmp

        cell_state, hidden = lstm.lstm_cell(tmp,cell_state,hidden)

    return outputs

### LSTM Dataset Code

In [5]:

class LSTMDataset(Dataset):
    def __init__(self, chunk_len=200, padded_chunks=False):
        # Character based dataset
        dataset_path = "./input.txt"
        # The tokens in the vocabulary (all_characters)
        # are just the printable characters of the string class
        self.all_characters = string.printable
        self.n_characters = len(self.all_characters)
        # Maps characters to indices
        self.char_dict = {x:i for i,x in enumerate(self.all_characters)}
        self.file, self.file_len = self.read_file(dataset_path)
        # Sequence length of the input
        self.chunk_len = chunk_len

    def read_file(self,filename):
        file = unidecode.unidecode(open(filename).read())
        return file, len(file)

    def char_tensor(self,in_str):
        # in_str - input sequence - String
        # Return one-hot encoded characters of in_str
        tensor = torch.zeros(len(in_str),self.n_characters).long()
        char_ind = [self.char_dict[c] for c in in_str]
        tensor[torch.arange(tensor.shape[0]),char_ind] = 1
        return tensor

    def __getitem__(self, idx):
        inp, target = self.get_random_text()
        return {"input":inp, "target":target}

    def __len__(self):
        return 10000

    def get_random_text(self):
        # Pick a random string of length self.chunk_len from the dataset
        start_index = np.random.randint(0, self.file_len - self.chunk_len)
        end_index = start_index + self.chunk_len + 1
        chunk = self.file[start_index:end_index]
        # One-hot encode the chosen string
        inp = self.char_tensor(chunk[:-1])
        # The target string is the same as the
        # input string but shifted by 1 character
        target = self.char_tensor(chunk[1:])
        inp = Variable(inp).cuda()
        target = Variable(target).cuda()
        return inp, target


### LSTM Training loop

With a correct implementation you should get sensible text generation results with the set parameters, however you should experiment with various parameters,
especially with the sequence length (chunk_len) used during training.

In [6]:
@torch.no_grad()
def sample_lstm_text(model, dataset, seed_text, gen_len=200, method="topk", top_k=5, **_):
    model.eval()

    x = dataset.char_tensor(seed_text).unsqueeze(0).float().to(device)
    _, (cell_state, hidden) = model(x)

    generated_ids = []

    for _ in range(gen_len):
        logits = model.proj(hidden)
        next_idx = sample_next_index(logits, method=method, top_k=top_k)

        next_onehot = torch.zeros(1, dataset.n_characters, device=device)
        next_onehot.scatter_(1, next_idx.view(1, 1), 1.0)

        generated_ids.append(next_idx.item())
        cell_state, hidden = model.lstm_cell(next_onehot, cell_state, hidden)

    return seed_text + ids_to_text(generated_ids, dataset)

def train_lstm_with_chunk_len(chunk_len, batch_size=256, hidden_dim=256, learning_rate=0.005, epochs=30, sample_every=5, sample_gen_len=300,):
    print("\n" + "=" * 100)
    print(f"Training LSTM with sequence length = {chunk_len}")
    print("=" * 100)

    dataset = LSTMDataset(chunk_len=chunk_len)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=0,
        drop_last=True,
    )

    model = LSTMSimple(chunk_len, dataset.n_characters, hidden_dim, dataset.n_characters, batch_size,).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    losses = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        pbar = tqdm(loader, desc=f"LSTM seq_len={chunk_len} epoch {epoch}/{epochs}", unit="batch")

        for batch in pbar:
            inputs = batch["input"].float().to(device)
            labels = batch["target"].float().to(device)
            targets = torch.argmax(labels, dim=2)

            optimizer.zero_grad()
            outputs, _ = model(inputs)

            loss = criterion(
                outputs.reshape(-1, outputs.size(-1)),
                targets.reshape(-1),
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=round(loss.item(), 4), lr=learning_rate)

        losses.append(total_loss / len(loader))

        if sample_every and epoch % sample_every == 0:
            seed = "O Romeo, wherefore art thou"
            print("\nTop-k sampling:")
            print(sample_lstm_text(model, dataset, seed, sample_gen_len, method="topk"))
            print("\nGreedy sampling:")
            print(sample_lstm_text(model, dataset, seed, sample_gen_len, method="greedy"))

    return {
        "chunk_len": chunk_len,
        "model": model,
        "dataset": dataset,
        "losses": losses,
    }

sequence_lengths = [64, 128, 256]
epoch_settings = [30, 60]

lstm_results = {}

for seq_len in sequence_lengths:
    for epochs in epoch_settings:
        key = f"seq_len_{seq_len}_epochs_{epochs}"

        lstm_results[key] = train_lstm_with_chunk_len(chunk_len=seq_len, batch_size=256, hidden_dim=256, learning_rate=0.005, epochs=epochs)

compare_trained_runs(lstm_results, sample_lstm_text, title="Comparison of LSTM models trained with different sequence lengths and epochs", gen_len=100,)




Training LSTM with sequence length = 64


LSTM seq_len=64 epoch 5/30: 100%|██████████| 39/39 [00:01<00:00, 20.86batch/s, loss=2.06, lr=0.005]



Top-k sampling:
O Romeo, wherefore art thour andersent,
And ard and and handes shate,
And mase will and allour thithilly at the thith to the the sulite weres soor the surtine.

CARIO:
Thar ancere me with shis har in at inde fires mand arthare ware, ald hen the lownther ta he all the pringe my thould madeser and henowere an in wire ant an and

Greedy sampling:
O Romeo, wherefore art thou hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the hand the hat the ha


LSTM seq_len=64 epoch 10/30: 100%|██████████| 39/39 [00:03<00:00, 12.27batch/s, loss=1.76, lr=0.005]



Top-k sampling:
O Romeo, wherefore art thou tord to be told and
That the sure is to the werends,
What so tood, sound and to makes senclanger head of stold
I'll stee hom sin,
Thish your be the wolld and she told, bucking of mine.

POLIXENES:
Hows, I muse to thou hade ho ame to heare,
When me tho gave me,
I shame thy buce the canded to have to

Greedy sampling:
O Romeo, wherefore art thou have the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the son the so


...


Top-k sampling:
O Romeo, wherefore art thou heapt it is assick?

Gentleman:
What thou seed thou dost,
Which not thy banish, see thought thou with all sweet son
That hats have stood with her being one souls,
There brother through the whitest be thy sister
As thou hadst be so stood with a fortune woman;
And then he stir not, as you have but hi

Greedy sampling:
O Romeo, wherefore art thou the prince of the
comfort of the prince of the prince of the world?

BAPTISTA:
I do not so stand and so the truth of heaven,
That we shall be the people and the world
That we will be the people and the world
That we will be the people and the world
That we will be the people and the world
That we w


LSTM seq_len=256 epoch 55/60: 100%|██████████| 39/39 [00:09<00:00,  3.92batch/s, loss=1.26, lr=0.005]



Top-k sampling:
O Romeo, wherefore art thou to me to bear
Whose done, were at that we am honour with
hangs; but now it stir a bawd of you. Too where,
As I will tell thee with my soul to the sea
To rear him but by a world would not take
A miston and thine ears, were no more.

PETRUCHIO:
Thou dost thou wilt call'd?

MENENIUS:
Thou shouldst mor

Greedy sampling:
O Romeo, wherefore art thou the sea to the court?

PETRUCHIO:
Why, then the state that we shall be the state,
And therefore thou art to be a sentence of the world.

KING RICHARD III:
What is the seat of your head?

AUFIDIUS:
I think you are the senate to the prince of mine own.

KING RICHARD III:
What say you to the consul?




LSTM seq_len=256 epoch 60/60: 100%|██████████| 39/39 [00:10<00:00,  3.83batch/s, loss=1.25, lr=0.005]



Top-k sampling:
O Romeo, wherefore art thou not speak?

DUKE VINCENTIO:
That, and weary as your sense as mine, and
the that hangs that thoughts to break his fortune in them,
And heard impatient lord, the matter well
Were thought an arms, to-night the seasting thing.
But what's to bear him think'st thou? a bot?
The sun, but all the manners wo

Greedy sampling:
O Romeo, wherefore art thou the country?

BAPTISTA:
Why, then, I will not so much to the common cause.

CORIOLANUS:
I will be the state of your souls,
And there the state of the house of York,
And that the seat of the house of York,
That she shall be the state of the house of York.

KING RICHARD II:
Then he hath been a word o

Comparison of LSTM models trained with different sequence lengths and epochs

########################################################################################################################
Example 1
###################################################################################################

# Task 2: Character generation transformer network implementation
Our simple transformer-like network will take as input a sequence of characters and predict the next character in the sequence. To ensure an efficient training procedure, masked attention modules will be used as in the [GPT model](https://s3-us-west-2.amazonaws.com/openai-assets/research-covers/language-unsupervised/language_understanding_paper.pdf).

For this task you must implement the Scaled dot product attention module and the Masked multi-head attention module. Both of these modules are described in the [Attention is all you need](https://arxiv.org/pdf/1706.03762.pdf) paper (See Figure 2 in the paper as well as Sections 3.2.1, 3.2.2 and 3.2.3). They are the core operations of transformers. As we will use our model for text generation also add the masking operation shown as (mask opt.) in Figure 2, implemented as AttentionMasking in the code.

**Implement the modules in the ScaledDotProductAttention class and the MultiHeadAttention class.**

Read the GPT paper and the Attention is all you need paper for a better understanding of the components. For a more high level overview, this [post](https://jalammar.github.io/illustrated-gpt2/) may also be helpful.

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        # Positional encoding adds the positional information to the
        # embedding. Without it the model would not be able to differentiate
        # between different characters orders such as between "dog" and "god".
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = 10000.0**(torch.arange(0,d_model,2).float()/d_model)
        print(div_term.shape)
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position / div_term)
        pe[:, 1::2] = torch.cos(position / div_term)
        pe = pe.unsqueeze(0)
        self.pe = pe.cuda()
        self.pe.requires_grad = False

    def forward(self, x):
        p = self.pe[:, :x.size(1)]
        return p

class AttentionMasking(nn.Module):
    def __init__(self, max_len):
        super(AttentionMasking, self).__init__()
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len))
                                     .view(1, 1, max_len, max_len))
    def forward(self,x):
        length = x.shape[-1]
        out = x.masked_fill(self.mask[:,:,:length,:length] == 0, float('-inf'))
        return out


class ScaledDotProductAttention(nn.Module):
    def __init__(self, max_len):
        super(ScaledDotProductAttention, self).__init__()
        self.softmax = nn.Softmax(dim=-1)
        # Multiply with an upper triangular
        # matrix of dimensions (length x length) after the scale operation
        # in Figure 2 of the paper.
        self.mask_opt = AttentionMasking(max_len)


    def forward(self,q,k,v):
        # length = number of input tokens
        batch_size,num_heads,length,num_neuron = k.size()
        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / math.sqrt(num_neuron)
        scores = self.mask_opt(scores)
        attn_weights = self.softmax(scores)
        out = torch.matmul(attn_weights, v)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, dim_model, num_neuron, n_head, max_len):
        super(MultiHeadAttention, self).__init__()
        self.n_head = n_head
        self.num_neuron = num_neuron

        self.w_q = nn.Linear(dim_model, n_head * num_neuron) # query projection 
        self.w_k = nn.Linear(dim_model, n_head * num_neuron) # key projection 
        self.w_v = nn.Linear(dim_model, n_head * num_neuron) # value projection
        self.attention = ScaledDotProductAttention(max_len) # compute attention scores
        self.w_concat = nn.Linear(n_head * num_neuron, dim_model) # project back to model dimension

    def split(self,tensor):
        batch_size, length, total_dim = tensor.size()
        # Reshape the tensor to enable the use in
        # the ScaledDotProductAttention module
        split_tensor = tensor.view(batch_size, length, self.n_head, self.num_neuron).transpose(1,2)
        return split_tensor

    def concat(self,tensor):
        batch_size, num_heads, length, num_neuron = tensor.size()
        # Reshape the tensor to its original size before the split operation.
        concat_tensor = tensor.transpose(1,2).contiguous().view(batch_size, length, self.n_head*self.num_neuron)
        return concat_tensor


    def forward(self, q, k, v):
        q = self.w_q(q)
        k = self.w_k(k)
        v = self.w_v(v)

        q = self.split(q)
        k = self.split(k)
        v = self.split(v)

        attn_out = self.attention(q, k, v)

        concat_out = self.concat(attn_out)
        out = self.w_concat(concat_out)

        return out



class PositionFeedForwardNet(nn.Module):
    def __init__(self, dim_model):
        super(PositionFeedForwardNet, self).__init__()
        self.ff_net1 = nn.Linear(dim_model, dim_model*4)
        self.ff_net2 = nn.Linear(dim_model*4, dim_model)
    def forward(self,x):
        ff_out = self.ff_net1(x)
        ff_out = torch.nn.functional.relu(ff_out)
        ff_out = self.ff_net2(ff_out)
        return ff_out


class TransformerBlock(nn.Module):
    def __init__(self, dim_model, num_neuron, n_head, max_len):
        super(TransformerBlock, self).__init__()
        self.mha = MultiHeadAttention(dim_model, num_neuron, n_head, max_len)
        self.l_norm = torch.nn.LayerNorm(dim_model)
        self.l_norm2 = torch.nn.LayerNorm(dim_model)
        self.ff_net = PositionFeedForwardNet(dim_model)
        # b, len_seq, n_head, num_neuron

    def forward(self, x):
      # A Transformer block as described in the
      # Attention is all you need paper. In Figure 1 the transformer
      # block is marked with a gray rectangle right of the text "Nx"
      _x = x
      mha1 = self.mha(x,x,x)
      lnorm = self.l_norm(_x+mha1)
      _x = lnorm
      ff_out = self.ff_net(lnorm)
      out = self.l_norm2(ff_out+_x)

      return out

class TransformerSimple(nn.Module):
    def __init__(self, seq_length, input_dim, output_dim,
                 batch_size):
        super(TransformerSimple, self).__init__()
        num_neuron = 64
        n_head = 8
        dim_model=256
        max_len = 512
        self.start_embedding = nn.Embedding(input_dim, dim_model)

        self.pos_embedding = PositionalEncoding(dim_model)

        # b x l x c*n_head
        self.t_block1 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block2 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block3 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block4 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block5 = TransformerBlock(dim_model, num_neuron, n_head, max_len)

        #self.out_layer_1 = nn.Linear(dim_model, dim_model)
        self.output_layer = nn.Linear(dim_model,output_dim)

    def forward(self,x):
      
      # embeds the input tensor from tokens to features
      s_emb = self.start_embedding(x)
      # add positional encodings
      p_emb = self.pos_embedding(s_emb)
      b_out = p_emb + s_emb
      # 5 transformer blocks
      b_out = self.t_block1(b_out)
      b_out = self.t_block2(b_out)
      b_out = self.t_block3(b_out)
      b_out = self.t_block4(b_out)
      b_out = self.t_block5(b_out)

      # output mapping to a classification of output tokens
      # for each token the network tries to predict the next token
      # based only on the previous tokens.
      out = self.output_layer(b_out)

      return out






In [8]:
class TextDataset(Dataset):
    def __init__(self, chunk_len=200, padded_chunks=False):
        # Character based dataset
        dataset_path = "./input.txt"
        # The tokens in the vocabulary (all_characters)
        # are just the printable characters of the string class
        self.all_characters = string.printable
        self.n_characters = len(self.all_characters)
        # Maps characters to indices
        self.char_dict = {x:i for i,x in enumerate(self.all_characters)}
        self.file, self.file_len = self.read_file(dataset_path)
        # Sequence length of the input
        self.chunk_len = chunk_len
        self.encoded_file = [self.char_dict[x] for x in self.file]

    def read_file(self,filename):
        file = unidecode.unidecode(open(filename).read())
        return file, len(file)

    def encode_text(self,in_str):
        # in_str - input sequence - String
        # Returns - in_str mapped to tokens in char_dict
        tensor = torch.LongTensor([self.char_dict[x] for x in in_str])
        return tensor

    def __getitem__(self, idx):
        inp, target = self.get_random_text()
        return {"input":inp, "target":target}

    def __len__(self):
        return 10000

    def get_random_text(self):
        # Pick a random string of length self.chunk_len from the dataset
        start_index = np.random.randint(0, self.file_len - self.chunk_len)
        end_index = start_index + self.chunk_len + 1
        chunk = self.encoded_file[start_index:end_index]
        # input_tokens - random sequence of tokens from the dataset
        input_tokens = torch.LongTensor(chunk[:-1])
        # target - input token sequence shifted by 1
        # the idea is to predict next token for each token in the input sequence
        # therefore if the input is [1,2,3,4] the target is [2,3,4,5]
        target = torch.LongTensor(chunk[1:])
        input_tokens = input_tokens.cuda()
        target = target.cuda()
        return input_tokens, target


## Character sampling

To generate text the network must predict the next character in a sequence, however networks do not produce a single character but rather estimate the likelihood for each possible character. Sampling characters from the network output can be done in different ways with common ones being the Greedy sampling process and Top-K sampling.

In the simple greedy sampling method the network takes a text prompt as input and generates an additional N tokens by always taking the token with the highest prediction score as the next token.

In the Top-K sampling, randomness is added to the sampling process as the network samples from K most likely predicitons at each step. This alleviates the problem of generative models repeating text but may generate incorrect text by sampling inappropriate tokens.


In [9]:
def topk_sampling_iter_transformer(model, x, num_chars, chunk_len, output_token):
    # x -- b x onehot_char
    # x = b x l
    outputs = torch.zeros((1,num_chars))
    inp = x

    for t in range(num_chars):
        # b x onehot_char
        output = model(inp.long())[0,-1:]
        #output = torch.softmax(output, dim=1)
        # b x 3
        output_vals, output_ind = torch.topk(output, 5, dim=1)
        # 3 -> int
        output_vals = torch.softmax(output_vals, dim=1)
        top_ind = torch.multinomial(output_vals[0], 1)[0]
        # int
        out_char_index = output_ind[0,top_ind]
        # int -> 1
        out_char_index = torch.ones(1).cuda() * out_char_index

        outputs[:,t] = out_char_index.item()
        if inp.shape[1] > chunk_len:
          inp = torch.cat((inp[:,1:], out_char_index.unsqueeze(0)), dim=1)
        else:
          inp = torch.cat((inp, out_char_index.unsqueeze(0)), dim=1)

    return outputs


def greedy_sampling_iter_transformer(model, x, num_chars, chunk_len, output_token):
    # x -- shape (batch, tokens in x)
    outputs = torch.zeros((1,num_chars))
    inp = x

    for t in range(num_chars):
        # b x l x onehot_char
        output = model(inp.long())[0,-1:]
        output = torch.softmax(output, dim=1)
        out_char_index = torch.argmax(output, dim=1)
        outputs[:,t] = out_char_index.item()
        if inp.shape[1] > chunk_len:
          inp = torch.cat((inp[:,1:], out_char_index.unsqueeze(0)), dim=1)
        else:
          inp = torch.cat((inp, out_char_index.unsqueeze(0)), dim=1)

    return outputs






## Transformer model training

With a correct implementation you should get sensible text generation results with the set parameters, however you should experiment with various parameters,
especially with the sequence length (chunk_len) used during training.

In [10]:
@torch.no_grad()
def sample_transformer_text(model, dataset, seed_text, gen_len=200, method="topk", top_k=5, chunk_len=None,):
    model.eval()

    if chunk_len is None:
        chunk_len = len(seed_text)

    inp = dataset.encode_text(seed_text).unsqueeze(0).long().to(device)
    generated_ids = []

    for _ in range(gen_len):
        context = inp[:, -chunk_len:]
        logits = model(context)[0, -1, :]

        next_idx = sample_next_index(logits, method=method, top_k=top_k)
        generated_ids.append(next_idx.item())

        inp = torch.cat([inp, next_idx.view(1, 1).to(device)], dim=1)

    return seed_text + ids_to_text(generated_ids, dataset)

def train_transformer_with_chunk_len(chunk_len, batch_size=256, learning_rate=0.0006, epochs=30, sample_every=5,sample_gen_len=300,):
    print("\n" + "=" * 100)
    print(f"Training Transformer with sequence length = {chunk_len}")
    print("=" * 100)

    dataset = TextDataset(chunk_len=chunk_len)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=0,
        drop_last=True,
    )

    model = TransformerSimple(
        chunk_len,
        dataset.n_characters,
        dataset.n_characters,
        batch_size,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    losses = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        pbar = tqdm(loader, desc=f"Transformer seq_len={chunk_len} epoch {epoch}/{epochs}", unit="batch")

        for batch in pbar:
            inputs = batch["input"].long().to(device)
            labels = batch["target"].long().to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(
                outputs.reshape(-1, outputs.size(-1)),
                labels.reshape(-1),
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=round(loss.item(), 4), lr=learning_rate)

        losses.append(total_loss / len(loader))

        if sample_every and epoch % sample_every == 0:
            seed = "O Romeo, wherefore art thou"
            print("\nTop-k sampling:")
            print(sample_transformer_text(model, dataset, seed, sample_gen_len, method="topk", chunk_len=chunk_len))
            print("\nGreedy sampling:")
            print(sample_transformer_text(model, dataset, seed, sample_gen_len, method="greedy", chunk_len=chunk_len))

    return {
        "chunk_len": chunk_len,
        "model": model,
        "dataset": dataset,
        "losses": losses,
    }

transformer_config = {
    64: {"epochs": 50},
    128: {"epochs": 50},
}

transformer_results = run_experiments(train_transformer_with_chunk_len, transformer_config)

compare_trained_runs(transformer_results, sample_transformer_text, title="Comparison of Transformer models trained with different sequence lengths", gen_len=200,)



Training Transformer with sequence length = 64
torch.Size([128])


Transformer seq_len=64 epoch 5/50: 100%|██████████| 39/39 [00:09<00:00,  4.20batch/s, loss=1.86, lr=0.0006]



Top-k sampling:
O Romeo, wherefore art thou the do friest,
But his betloouds to thigh hash broth heasst
And spreect to the crust and and and man all
The trengeed is bringnentio of his thee stan offffartel,
This wonds the than tous than were a bloowss and
To man to the dof by this shonghe there, all
Wherow and this his winght our his tanke
We

Greedy sampling:
O Romeo, wherefore art thou hand the stand the stranges
That the shall the stand the shall the stand the shall the stand
The shall the stand the shall the stand the shall the stand
The shall the stand the shall the stand the shall the stand
The shall the stand the shall the stand the shall the stand
The shall the stand the sh


Transformer seq_len=64 epoch 10/50: 100%|██████████| 39/39 [00:09<00:00,  4.19batch/s, loss=1.52, lr=0.0006]



Top-k sampling:
O Romeo, wherefore art thought of this? house that they weere
To soul, when we mine along.

BUCKINGHAM:
Where we will, I here shame to the crown to them wounds.
Whose have we willl'ed for my light a cast of anclose,
As true to him he best my sonse where so.

KING RICHARD II:
He will head bless my besechiong of the said;
And a

Greedy sampling:
O Romeo, wherefore art thou shall be the strange
That the shall be the see the see the see the see
That was the see the see the see the see the see the see
That was the see the see the see the see the see the see
That was the see the see the see the see the see the see
That was the see the see the see the see the see the see



...


Top-k sampling:
O Romeo, wherefore art thou advertisle,
Thy father was, as the first sauce of heaven dwell
But on thy side. They are under some griefs.

PAULINA:
Trank up the gates for the whinds ne'er I do take in hangmen.

LADY ANNE:
If I'll say thou discant before thy deed with spleen.

GLOUCESTER:
To thee ginstry Paulina's day that years

Greedy sampling:
O Romeo, wherefore art thou speak'st thy colours.

LEONTES:
Thou hast a thousand fools for the world that word
That word the garden conduct his face.

PAULINA:
I know not what noisome struck to the people.

LEONTES:
What news is this thine, that thou shouldst know the
shalt be the seat of this factious in the world,
Thou shou


Transformer seq_len=128 epoch 45/50: 100%|██████████| 39/39 [00:15<00:00,  2.48batch/s, loss=0.955, lr=0.0006]



Top-k sampling:
O Romeo, wherefore art thou she weak.
I'll harbour thee tilf that may blood loe:
Some we will not there; what thou wert.

GREMIO:
Father, and thy friends no less!
Go where you are rash that she would show
you must not say, lain with me to thee fight;
Thy brave in good two blemish measure from
The that come of thy world will b

Greedy sampling:
O Romeo, wherefore art thou shalt see, thou shalt stay thy brother,
Thou love my lord, thou hast not so do the time.

GLOUCESTER:
The common counsel, where they shall be truth
To the substitution of thy strengthening blood,
That they shall be thy state, thy love, thy love,
That thou shalt not seem to be thy brother,
As thou l


Transformer seq_len=128 epoch 50/50: 100%|██████████| 39/39 [00:15<00:00,  2.48batch/s, loss=0.862, lr=0.0006]



Top-k sampling:
O Romeo, wherefore art thou hast
From married that ha't as make as was an as
Sometime coursely, as I am. I did hamary
As it as the haund the wretched with that an entracted
In permitted present at the city, and with him
Be he hugh powers.

DUKE VINCENTIO:
What you do that?

LUCIO:
Is the thirthous?

ELBOW:
Marry, sir, I am at

Greedy sampling:
O Romeo, wherefore art thou hast spoken his
At that thy great company, the time and patient
In manage as lies. Thy mother was not incensed;
For we have show'd it, we should have heard it.

KING RICHARD II:
Now, by the holy of the world, the house of them,
The could stay and look that way to the proudest by the wall.

GRUMIO:


Comparison of Transformer models trained with different sequence lengths

########################################################################################################################
Example 1
#######################################################################################################